In [ ]:
# import dependencies
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

Data loaded! Generating GIF...
GIF saved successfully as 'aerodynamics_convergence.gif'!


In [ ]:
# MLP history
with open('pickle/history.pkl', 'rb') as file:
    historyMLP = pickle.load(file)

# gbt data
with open('pickle/gbt_history.pkl', 'rb') as file:
    gbt_data = pickle.load(file)

print("Data loaded!")


Data loaded!


## MLP Convergence Plot

In [ ]:
# extract data
parameters = ["CL", "CD", "CY", "Cl", "Cm", "Cn"]
train_maes = np.array(historyMLP["trainMAE_raw"])
val_maes = np.array(historyMLP["validateMAE_raw"])
epochs = historyMLP["epoch"]

# set up figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('MLP Convergence: Training vs Validation MAE over Epochs', fontsize=18)
axes = axes.flatten()

train_lines = {}
val_lines = {}

for i, param in enumerate(parameters):
    ax = axes[i]
    val_history_single = val_maes[:, i]
    
    # intitlize empty lines
    train_lines[param], = ax.plot([], [], label='Train MAE', color='blue', linewidth=2)
    val_lines[param], = ax.plot([], [], label='Validation MAE', color='red', linestyle='--', linewidth=2)
    
    ax.set_xlim(0, max(epochs))
    ax.set_ylim(bottom=0, top=max(val_history_single[10:]) * 1.2)

    ax.set_title(f"{param} Convergence", fontsize=14)
    ax.set_xlabel('Epochs', fontsize=12)
    ax.set_ylabel('Mean Absolute Error', fontsize=12)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(top=0.9)

# animation function
frame_step = 5 
frames_to_render = np.arange(0, len(epochs), frame_step)

def update(frame):
    for i, param in enumerate(parameters):
        current_epochs = epochs[:frame]
        current_train = train_maes[:frame, i]
        current_val = val_maes[:frame, i]
        
        train_lines[param].set_data(current_epochs, current_train)
        val_lines[param].set_data(current_epochs, current_val)
        
    return list(train_lines.values()) + list(val_lines.values())

# create and save
anim = animation.FuncAnimation(fig, update, frames=frames_to_render, interval=50, blit=True)
anim.save('aerodynamics_convergence.gif', writer='pillow', fps=20)

print("GIF saved successfully as 'aerodynamics_convergence.gif'!")
plt.close()

## GBT Convergence

In [ ]:
all_train_errors = gbt_data["train_errors"]
all_val_errors = gbt_data["val_errors"]
stages = gbt_data["stages"]
parameters = ["CL", "CD", "CY", "Cl", "Cm", "Cn"]

print("Data loaded! Setting up plot with updated spacing...")

# --- 2. SET UP THE FIGURE AND FIXED LIMITS ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('GBT Convergence: Training vs Validation MSE over Boosting Stages', fontsize=18)
axes = axes.flatten()

# Dictionaries to hold the line objects
train_lines = {}
val_lines = {}

for i, param in enumerate(parameters):
    ax = axes[i]
    v_err = np.array(all_val_errors[i])
    
    # --- FIX START ---
    # To fix y-limits to prevent jitter AND create space at the bottom,
    # we calculate limits based on the final data, skipping high initial values.

    # 1. Skip first 10 trees to find typical "converged" min/max (huge drops distort scale)
    if len(v_err) > 10:
        v_baseline = v_err[10:]
    else:
        v_baseline = v_err

    baseline_min = v_baseline.min()
    baseline_max = v_baseline.max()
    baseline_range = baseline_max - baseline_min

    # Add safety check for zero range (very rare, but possible e.g., error flatlines)
    if baseline_range == 0:
        baseline_range = 0.01 * baseline_max + 1e-6

    # 2. Define Limits: Add space (padding) relative to the Converged region's range.

    # New Bottom: Set slightly BELOW baseline_min to give that desired space.
    y_bottom = baseline_min - (0.1 * baseline_range) # 10% range space below

    # Maintain existing high top margin (adds 20% space above highest point after initial drop)
    if len(v_err) > 5:
        y_top = max(v_err[5:]) * 1.2
    else:
        y_top = max(v_err) * 1.2

    # Lock these fixed axes so they don't bounce around during animation
    ax.set_ylim(bottom=y_bottom, top=y_top)
    # --- FIX END ---

    # Lock the stages axis
    ax.set_xlim(1, len(stages))
    
    # Initialize empty lines that the update function will populate
    train_lines[param], = ax.plot([], [], label='Train MSE', color='blue', linewidth=2)
    val_lines[param], = ax.plot([], [], label='Validation MSE', color='red', linestyle='--', linewidth=2)

    ax.set_title(f"{param} Convergence", fontsize=14)
    ax.set_xlabel('Boosting Iterations (Trees)', fontsize=12)
    ax.set_ylabel('Mean Squared Error', fontsize=12)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- 3. ANIMATION UPDATE FUNCTION ---
# This remains the same
frame_step = 2 
frames_to_render = np.arange(1, len(stages) + 1, frame_step)

def update(frame):
    for i, param in enumerate(parameters):
        current_stages = stages[:frame]
        current_train = all_train_errors[i][:frame]
        current_val = all_val_errors[i][:frame]
        
        train_lines[param].set_data(current_stages, current_train)
        val_lines[param].set_data(current_stages, current_val)
        
    return list(train_lines.values()) + list(val_lines.values())

# --- 4. RENDER AND SAVE ---
print("Generating GIF...")
anim = animation.FuncAnimation(fig, update, frames=frames_to_render, interval=50, blit=True)
anim.save('gbt_convergence.gif', writer='pillow', fps=20)

print("GIF saved successfully as 'gbt_convergence.gif'!")
plt.close()

Data loaded! Setting up plot with updated spacing...
Generating GIF...
GIF saved successfully as 'gbt_convergence.gif'!
